In [255]:
import pandas as pd
import numpy as np
import warnings
from tables import NaturalNameWarning
warnings.filterwarnings('ignore', category=NaturalNameWarning)

In [263]:
GB_U_Values = pd.read_excel("../Data/Tabula_U_Values.xlsx",index_col=0)

In [261]:
Materials_Ambience = pd.read_excel("../Data/Materials.xlsx", index_col=0)
ambience = pd.read_excel("../Data/AmBIENCe_Geometry_Constructions.xlsx")

# Finding the materials and their thicknesses within construction elements in the English Housing Stock

## Background
For the English housing stock, we know the U-values of each constriction element (wall, floors etc) of each residential archetype from TABULA data. However, we do not know how those elements are 'built-up'. Is it a brick wall? Or a brick wall with insulation? What kind of brick and what kind of insulation etc.?

Therefore, we need a way of determining the build-up from the U-Value. The forumula for the U-Value is:

$$ U = \frac{1}{R_{si}+\frac{d_{mat}}{\gamma_{mat}}+ \frac{d_{ins}}{\gamma_{ins}}+R_{se}}$$

where $R_{si} , R_{se} , d_{mat} , \gamma_{mat} , d_{ins} , \gamma_{ins}$ , are the thermal resistances of the internal and external surfaces, the thickness and thermal conductivity of the bulk construction material and the thickness and thermal conductivity of the insulation material respectively. 

As noted, we know the U-Values but we also have a list of common construction and insulation materials from AmBIENCe. In addition, the thermal resistances of the surfaces are also commonly reported in the literature. So theoretically, the only unknown is the thickness of the construction material. 

The U-Values of the English housing stock are:

In [264]:
GB_U_Values

,Code_BuildingVariant,U_Roof_1,U_Wall_1,U_Floor_1,U_Window_1,U_Door_1,d_Insulation_Roof_1,d_Insulation_Wall_1,d_Insulation_Floor_1,Hotmaps Mapping
0,GB.ENG.AB.03.Gen.ReEx.001.001,2.30,2.10,0.45,4.80,3.0,0.000,0,0,NaN
1,GB.ENG.AB.04.Gen.ReEx.001.001,1.50,1.60,0.45,3.10,1.8,0.012,0,0,NaN
2,GB.ENG.AB.07.Gen.ReEx.001.001,0.25,0.28,0.25,1.85,1.8,0.250,0,0,NaN
3,GB.ENG.MFH.01.Gen.ReEx.001.001,2.30,2.10,0.45,4.80,3.0,0.000,0,0,NaN
4,GB.ENG.MFH.02.Gen.ReEx.001.001,2.30,2.10,0.45,4.80,3.0,0.000,0,0,NaN
5,GB.ENG.MFH.03.Gen.ReEx.001.001,2.30,1.60,0.45,4.80,3.0,0.000,0,0,NaN
6,GB.ENG.MFH.04.Gen.ReEx.001.001,1.50,1.60,0.45,3.10,1.8,0.012,0,0,NaN
7,GB.ENG.MFH.05.Gen.ReEx.001.001,0.40,1.60,0.45,3.10,1.8,0.100,0,0,NaN
8,GB.ENG.MFH.06.Gen.ReEx.001.001,0.35,1.60,0.45,3.10,1.8,0.150,0,0,NaN
9,GB.ENG.MFH.07.Gen.ReEx.001.001,0.25,0.35,0.25,1.85,1.8,0.250,0,0,NaN


and the materials used within the AmBIENCe database are:

In [265]:
Materials_Ambience

,Thermal_Conductivity,Density,Specific_Heat_Capacity,Material Type
Material,,,,
Cement fibre slabs shredded wood,0.080,350,1030,Insulation
Asbestos fibre,0.060,640,840,Insulation
Perlite board expanded,0.052,16,1260,Insulation
Rock wool,0.034,200,710,Insulation
Urea formaldehyde resin foam,0.054,14,147,Insulation
Mineral wool,0.042,12,1030,Insulation
Polystyrene expanded,0.035,23,1470,Insulation
Polyurethane foam,0.025,30,1400,Insulation
Polyisocyanurate aged with facers,0.020,25,1470,Insulation


the materials which are not labelled as insulation materials are considered to be the construction materials 

Therefore, it is possible to estimate the thickness of bulk construction material  by rearranging equation one to solve for $d_{cons}$

$$d_{cons}=\gamma_{cons}(\frac{1}{U}-R_{si}-R_{se}-\frac{d_{ins}}{\gamma_{ins}})$$

So for every construction material listed in the AmBIENCe materials database we can create a build-up with the insulation material and see what the required thickness of the construction material would need to be to achieve the U-value stipulated by TABULA. 

And to narrow down sensible build-ups we can look at the minimum and maximum thickness values of the construction materials used within the AmBIENCe database. 

The following process will go as follows; identify:
- the minimum and maximum thickness each construction material attained in each element
- the thickness required of a construction material to achieve a specific U-value 

## Min/Max

### Walls - minimum and maximum material thicknesses

In [267]:
wall_thickness_range = pd.DataFrame()
wall_materials = ambience["REFERENCE BUILDING WALL MATERIAL"].unique()

for material in wall_materials:
    wall_thickness = ambience[ambience["REFERENCE BUILDING WALL MATERIAL"] == material]
    wall_thickness = wall_thickness["REFERENCE BUILDING WALL MATERIAL THICKNESS (m)"]
    min_wall_thickness = wall_thickness.min()
    max_wall_thickness = wall_thickness.max()
    s = pd.Series(["Wall",material, min_wall_thickness, max_wall_thickness], index=["Element","Material", "Minimum Thickness", "Maximum Thickness"])
    wall_thickness_range = pd.concat([wall_thickness_range, s], axis = 1)
    
wall_thickness_range = wall_thickness_range.T


In [268]:
wall_thickness_range

,Element,Material,Minimum Thickness,Maximum Thickness
0,Wall,Precast concrete (dense) (exposed),0.110549,0.995
0,Wall,Brick fired clay 1920,0.008497,0.999701
0,Wall,Brick fired clay 1280,0.964623,0.989401
0,Wall,Brick fired clay 1600,0.545889,0.987383
0,Wall,Brick fired clay 2080,0.101871,0.101871
0,Wall,Precast concrete (dense) (protected),0.997773,0.997773
0,Wall,Oak beech ash walnut,0.10661,0.653827
0,Wall,Granite red,0.693514,0.998456
0,Wall,Cast concrete 2000,0.982704,0.982704
0,Wall,Sandstone,0.985735,0.990935


### Floors - minimum and maximum material thicknesses

In [271]:
floor_materials = ambience["REFERENCE BUILDING FLOOR MATERIAL"].unique()
floor_thickness_range = pd.DataFrame()

for material in floor_materials:
    thickness = ambience[ambience["REFERENCE BUILDING FLOOR MATERIAL"] == material]
    thickness = thickness["REFERENCE BUILDING FLOOR MATERIAL THICKNESS (m)"]
    min_thickness = thickness.min()
    max_thickness = thickness.max()
    s = pd.Series(["Floor",material, min_thickness, max_thickness], index=["Element","Material", "Minimum Thickness", "Maximum Thickness"])
    floor_thickness_range = pd.concat([floor_thickness_range, s], axis = 1)
    
floor_thickness_range = floor_thickness_range.T


In [272]:
floor_thickness_range

,Element,Material,Minimum Thickness,Maximum Thickness
0,Floor,Precast concrete (dense) (exposed),0.10475,1.495376
0,Floor,Precast concrete (dense) (protected),0.736089,1.496783
0,Floor,Cast concrete 2000,1.473468,1.499829
0,Floor,Concrete block (dense) (protected),0.100632,0.100632
0,Floor,Brick fired clay 1920,0.182801,0.9423
0,Floor,Limestone,1.479512,1.479512
0,Floor,Oak beech ash walnut,0.100664,0.546675
0,Floor,Granite red,0.327356,1.499965


### Roof  - minimum and maximum material thicknesses

In [273]:
roof_materials = ambience["REFERENCE BUILDING ROOF MATERIAL"].unique()
roof_thickness_range = pd.DataFrame()

for material in roof_materials:
    thickness = ambience[ambience["REFERENCE BUILDING ROOF MATERIAL"] == material]
    thickness = thickness["REFERENCE BUILDING ROOF MATERIAL THICKNESS (m)"]
    min_thickness = thickness.min()
    max_thickness = thickness.max()
    s = pd.Series(["Roof",material, min_thickness, max_thickness], index=["Element","Material", "Minimum Thickness", "Maximum Thickness"])
    roof_thickness_range = pd.concat([roof_thickness_range, s], axis = 1)
    
roof_thickness_range = roof_thickness_range.T

In [274]:
roof_thickness_range

,Element,Material,Minimum Thickness,Maximum Thickness
0,Roof,Cast concrete 2000,0.987066,0.999818
0,Roof,Precast concrete (dense) (exposed),0.1025,0.998239
0,Roof,Precast concrete (dense) (protected),0.982714,0.995211
0,Roof,Oak beech ash walnut,0.100236,0.979535
0,Roof,Brick fired clay 1920,0.213427,0.299348
0,Roof,Maple oak and similar hardwoods,0.9376,0.966776


### Combining these min/max dataframes into a multi-index dataframe 

In [275]:
min_max = pd.concat([roof_thickness_range, wall_thickness_range,floor_thickness_range])
min_max.set_index(['Element', 'Material'], inplace=True)

In [276]:
min_max

Minimum Thickness  \
Element Material                                                 
Roof    Cast concrete 2000                            0.987066   
        Precast concrete (dense) (exposed)              0.1025   
        Precast concrete (dense) (protected)          0.982714   
        Oak beech ash walnut                          0.100236   
        Brick fired clay 1920                         0.213427   
        Maple oak and similar hardwoods                 0.9376   
Wall    Precast concrete (dense) (exposed)            0.110549   
        Brick fired clay 1920                         0.008497   
        Brick fired clay 1280                         0.964623   
        Brick fired clay 1600                         0.545889   
        Brick fired clay 2080                         0.101871   
        Precast concrete (dense) (protected)          0.997773   
        Oak beech ash walnut                           0.10661   
        Granite red                                   0.693514   
        Cast concrete 2000                            0.982704   
        Sandstone                                     0.985735   
        Limestone                                     0.985183   
Floor   Precast concrete (dense) (exposed)             0.10475   
        Precast concrete (dense) (protected)          0.736089   
        Cast concrete 2000                            1.473468   
        Concrete block (dense) (protected)            0.100632   
        Brick fired clay 1920                         0.182801   
        Limestone                                     1.479512   
        Oak beech ash walnut                          0.100664   
        Granite red                                   0.327356   

                                             Maximum Thickness  
Element Material                                                
Roof    Cast concrete 2000                            0.999818  
        Precast concrete (dense) (exposed)            0.998239  
        Precast concrete (dense) (protected)          0.995211  
        Oak beech ash walnut                          0.979535  
        Brick fired clay 1920                         0.299348  
        Maple oak and similar hardwoods               0.966776  
Wall    Precast concrete (dense) (exposed)               0.995  
        Brick fired clay 1920                         0.999701  
        Brick fired clay 1280                         0.989401  
        Brick fired clay 1600                         0.987383  
        Brick fired clay 2080                         0.101871  
        Precast concrete (dense) (protected)          0.997773  
        Oak beech ash walnut                          0.653827  
        Granite red                                   0.998456  
        Cast concrete 2000                            0.982704  
        Sandstone                                     0.990935  
        Limestone                                     0.985394  
Floor   Precast concrete (dense) (exposed)            1.495376  
        Precast concrete (dense) (protected)          1.496783  
        Cast concrete 2000                            1.499829  
        Concrete block (dense) (protected)            0.100632  
        Brick fired clay 1920                           0.9423  
        Limestone                                     1.479512  
        Oak beech ash walnut                          0.546675  
        Granite red                                   1.499965

## Identifying the required thickness of the materials to match with an archetypes U value

It is necessary to create use the min_max dataframe indices to create a new dataframe where the columns are now the possible insulation materials needed to create the build-up. Below is just creating this dataframe and filling it with random intergers in the place where the thickness should be

In [278]:
construction_elements = ["Roof", "Wall", "Floor"]
insulation_materials = Materials_Ambience[Materials_Ambience["Material Type"]=="Insulation"].index

num_roof_materials = [construction_elements[0]]*len(roof_materials)
num_wall_materials = [construction_elements[1]]*len(wall_materials)
num_floor_materials = [construction_elements[2]]*len(floor_materials)
num_materials = num_roof_materials+num_wall_materials+num_floor_materials

construction_materials = np.append(roof_materials, wall_materials)
construction_materials = np.append(construction_materials, floor_materials)

arrays = [num_materials, construction_materials]

options = pd.DataFrame(np.random.randint(1, 200, size=(len(num_materials), len(insulation_materials))), index=arrays, columns=insulation_materials)
options

Material                                    Cement fibre slabs shredded wood  \
Roof  Cast concrete 2000                                                  31   
      Precast concrete (dense) (exposed)                                 101   
      Precast concrete (dense) (protected)                                10   
      Oak beech ash walnut                                               180   
      Brick fired clay 1920                                               35   
      Maple oak and similar hardwoods                                      3   
Wall  Precast concrete (dense) (exposed)                                  36   
      Brick fired clay 1920                                               30   
      Brick fired clay 1280                                              177   
      Brick fired clay 1600                                               88   
      Brick fired clay 2080                                              167   
      Precast concrete (dense) (protected)                                44   
      Oak beech ash walnut                                                21   
      Granite red                                                        159   
      Cast concrete 2000                                                  21   
      Sandstone                                                          156   
      Limestone                                                           48   
Floor Precast concrete (dense) (exposed)                                  83   
      Precast concrete (dense) (protected)                               173   
      Cast concrete 2000                                                  92   
      Concrete block (dense) (protected)                                  96   
      Brick fired clay 1920                                               18   
      Limestone                                                          149   
      Oak beech ash walnut                                                 6   
      Granite red                                                        162   

Material                                    Asbestos fibre  \
Roof  Cast concrete 2000                                19   
      Precast concrete (dense) (exposed)                34   
      Precast concrete (dense) (protected)              18   
      Oak beech ash walnut                              88   
      Brick fired clay 1920                            160   
      Maple oak and similar hardwoods                   56   
Wall  Precast concrete (dense) (exposed)                72   
      Brick fired clay 1920                              2   
      Brick fired clay 1280                            141   
      Brick fired clay 1600                            160   
      Brick fired clay 2080                             92   
      Precast concrete (dense) (protected)             128   
      Oak beech ash walnut                              81   
      Granite red                                       42   
      Cast concrete 2000                               127   
      Sandstone                                         41   
      Limestone                                         67   
Floor Precast concrete (dense) (exposed)               112   
      Precast concrete (dense) (protected)               8   
      Cast concrete 2000                                57   
      Concrete block (dense) (protected)               168   
      Brick fired clay 1920                             70   
      Limestone                                        133   
      Oak beech ash walnut                             140   
      Granite red                                      103   

Material                                    Perlite board expanded  Rock wool  \
Roof  Cast concrete 2000                                        73         55   
      Precast concrete (dense) (exposed)                        99         70   
      Precast concrete (dense) (protected)                     

the formula for calculating the material thickness is:

In [279]:
def calculate_material_thickness(mat_therm_cond, U_Value, R_si, R_se, ins_thickness, ins_therm_cond):
    return mat_therm_cond * ((1/U_Value)-R_si-R_se-(ins_thickness/ins_therm_cond))

and so the required thicknesses are (nan is used when the thickness falls outside of the min-max range):

note: if all row entries are the same value that means that TABULA says no insulation material is used in the build-up

In [280]:
for building_code, archetype in GB_U_Values["Code_BuildingVariant"].iteritems():
    for index, row in options.iterrows():
        for insulation in insulation_materials:  
            

            construction_element = index[0]
            construction_material = index[1]

            U_value = GB_U_Values["U_"+construction_element+"_1"].iloc[building_code]
            insulation_thickness = GB_U_Values["d_Insulation_"+construction_element+"_1"].iloc[building_code]

            material_properties = Materials_Ambience[Materials_Ambience.index==construction_material]
            material_thermal_conductivity = material_properties["Thermal_Conductivity"].values[0]

            insulation_properties = Materials_Ambience[Materials_Ambience.index==insulation]
            insulation_thermal_conductivity = insulation_properties["Thermal_Conductivity"].values[0]


            R_se = 0.04

            if construction_element == "Floor":
                R_si = 0.17
            elif construction_element == "Wall":
                R_si = 0.13
            elif construction_element == "Roof":
                R_si = 0.10
            else:
                print("Incorrect construction element")  


            value = calculate_material_thickness(mat_therm_cond=material_thermal_conductivity, 
                                                           U_Value=U_value, 
                                                           R_si=R_si, 
                                                           R_se=R_se, 
                                                           ins_thickness=insulation_thickness, 
                                                           ins_therm_cond=insulation_thermal_conductivity)



            if value >= min_max.loc[index][0] and value <= min_max.loc[index][1]:
                df.loc[(index[0],index[1]), insulation] = value
            else:
                df.loc[(index[0],index[1]), insulation] = float("nan") 
                
                
            #df = df.dropna(how="all")
            df.to_hdf('data.h5', key="t"+archetype, mode='a') 
            
              
            
        

For the archetype of GB.ENG.TH.08.Gen.ReEx.001.001, which is an English (ENG) Terraced House and (TH) is the 8th (08) generation of all terraced houses in England, the U-values indicate that the material build-up should be:

In [257]:
df

Material                                    Cement fibre slabs shredded wood  \
Roof  Oak beech ash walnut                                           0.68655   
      Precast concrete (dense) (exposed)                                 NaN   
      Cast concrete 2000                                                 NaN   
      Precast concrete (dense) (protected)                               NaN   
      Brick fired clay 1920                                              NaN   
      Maple oak and similar hardwoods                                    NaN   
Wall  Precast concrete (dense) (exposed)                                 NaN   
      Brick fired clay 1920                                              NaN   
      Brick fired clay 1280                                              NaN   
      Brick fired clay 1600                                              NaN   
      Brick fired clay 2080                                              NaN   
      Precast concrete (dense) (protected)                               NaN   
      Oak beech ash walnut                                               NaN   
      Granite red                                                        NaN   
      Cast concrete 2000                                                 NaN   
      Sandstone                                                          NaN   
      Limestone                                                          NaN   
Floor Precast concrete (dense) (exposed)                                 NaN   
      Precast concrete (dense) (protected)                               NaN   
      Cast concrete 2000                                                 NaN   
      Concrete block (dense) (protected)                                 NaN   
      Brick fired clay 1920                                              NaN   
      Limestone                                                          NaN   
      Oak beech ash walnut                                               NaN   
      Granite red                                                        NaN   

Material                                    Asbestos fibre  \
Roof  Oak beech ash walnut                        0.446967   
      Precast concrete (dense) (exposed)               NaN   
      Cast concrete 2000                               NaN   
      Precast concrete (dense) (protected)             NaN   
      Brick fired clay 1920                            NaN   
      Maple oak and similar hardwoods                  NaN   
Wall  Precast concrete (dense) (exposed)               NaN   
      Brick fired clay 1920                            NaN   
      Brick fired clay 1280                            NaN   
      Brick fired clay 1600                            NaN   
      Brick fired clay 2080                            NaN   
      Precast concrete (dense) (protected)             NaN   
      Oak beech ash walnut                             NaN   
      Granite red                                      NaN   
      Cast concrete 2000                               NaN   
      Sandstone                                        NaN   
      Limestone                                        NaN   
Floor Precast concrete (dense) (exposed)               NaN   
      Precast concrete (dense) (protected)             NaN   
      Cast concrete 2000                               NaN   
      Concrete block (dense) (protected)               NaN   
      Brick fired clay 1920                            NaN   
      Limestone                                        NaN   
      Oak beech ash walnut                             NaN   
      Granite red                                      NaN   

Material                                    Perlite board expanded  Rock wool  \
Roof  Oak beech ash walnut                                0.299531        NaN   
      Precast concrete (dense) (exposed)                       NaN        NaN   
      Cast concrete 2000                                       

which when filtered for only real values:

In [285]:
df.dropna(how="all")

Material                                 Cement fibre slabs shredded wood  \
Roof Oak beech ash walnut                                         0.68655   
     Precast concrete (dense) (exposed)                               NaN   

Material                                 Asbestos fibre  \
Roof Oak beech ash walnut                      0.446967   
     Precast concrete (dense) (exposed)             NaN   

Material                                 Perlite board expanded  Rock wool  \
Roof Oak beech ash walnut                              0.299531        NaN   
     Precast concrete (dense) (exposed)                     NaN        NaN   

Material                                 Urea formaldehyde resin foam  \
Roof Oak beech ash walnut                                    0.340485   
     Precast concrete (dense) (exposed)                           NaN   

Material                                 Mineral wool  Polystyrene expanded  \
Roof Oak beech ash walnut                         NaN                   NaN   
     Precast concrete (dense) (exposed)      0.245886                   NaN   

Material                                 Polyurethane foam  \
Roof Oak beech ash walnut                              NaN   
     Precast concrete (dense) (exposed)                NaN   

Material                                 Polyisocyanurate aged with facers  
Roof Oak beech ash walnut                                              NaN  
     Precast concrete (dense) (exposed)                                NaN

so clearly there is an issue. It appears that no pairing of construction material and insulation can lead to an appropriate build-up for a wall or flooring for this archetype. 

The next steps are to get in touch with AmBIENCe and see how they managed to reconcile stuff like this. 

### Try it out for the other archetypes

In [282]:
arches = GB_U_Values["Code_BuildingVariant"]
arch = arches.iloc[-15]
arch

'GB.ENG.SFH.02.Gen.ReEx.001.001'

In [283]:
fd = pd.read_hdf('data.h5',arch)
fd

Material                                    Cement fibre slabs shredded wood  \
Roof  Oak beech ash walnut                                               NaN   
      Precast concrete (dense) (exposed)                            0.459861   
      Cast concrete 2000                                                 NaN   
      Precast concrete (dense) (protected)                               NaN   
      Brick fired clay 1920                                         0.265304   
      Maple oak and similar hardwoods                                    NaN   
Wall  Precast concrete (dense) (exposed)                            0.477657   
      Brick fired clay 1920                                         0.275571   
      Brick fired clay 1280                                              NaN   
      Brick fired clay 1600                                              NaN   
      Brick fired clay 2080                                              NaN   
      Precast concrete (dense) (protected)                               NaN   
      Oak beech ash walnut                                               NaN   
      Granite red                                                   0.762414   
      Cast concrete 2000                                                 NaN   
      Sandstone                                                          NaN   
      Limestone                                                          NaN   
Floor Precast concrete (dense) (exposed)                                 NaN   
      Precast concrete (dense) (protected)                               NaN   
      Cast concrete 2000                                                 NaN   
      Concrete block (dense) (protected)                                 NaN   
      Brick fired clay 1920                                              NaN   
      Limestone                                                          NaN   
      Oak beech ash walnut                                          0.271144   
      Granite red                                                        NaN   

Material                                    Asbestos fibre  \
Roof  Oak beech ash walnut                             NaN   
      Precast concrete (dense) (exposed)          0.459861   
      Cast concrete 2000                               NaN   
      Precast concrete (dense) (protected)             NaN   
      Brick fired clay 1920                       0.265304   
      Maple oak and similar hardwoods                  NaN   
Wall  Precast concrete (dense) (exposed)          0.477657   
      Brick fired clay 1920                       0.275571   
      Brick fired clay 1280                            NaN   
      Brick fired clay 1600                            NaN   
      Brick fired clay 2080                            NaN   
      Precast concrete (dense) (protected)             NaN   
      Oak beech ash walnut                             NaN   
      Granite red                                 0.762414   
      Cast concrete 2000                               NaN   
      Sandstone                                        NaN   
      Limestone                                        NaN   
Floor Precast concrete (dense) (exposed)               NaN   
      Precast concrete (dense) (protected)             NaN   
      Cast concrete 2000                               NaN   
      Concrete block (dense) (protected)               NaN   
      Brick fired clay 1920                            NaN   
      Limestone                                        NaN   
      Oak beech ash walnut                        0.271144   
      Granite red                                      NaN   

Material                                    Perlite board expanded  Rock wool  \
Roof  Oak beech ash walnut                                     NaN        NaN   
      Precast concrete (dense) (exposed)                  0.459861   0.459861   
      Cast concrete 2000                                       

In [284]:
fd.dropna(how="all")

Material                                  Cement fibre slabs shredded wood  \
Roof  Precast concrete (dense) (exposed)                          0.459861   
      Brick fired clay 1920                                       0.265304   
Wall  Precast concrete (dense) (exposed)                          0.477657   
      Brick fired clay 1920                                       0.275571   
      Granite red                                                 0.762414   
Floor Oak beech ash walnut                                        0.271144   

Material                                  Asbestos fibre  \
Roof  Precast concrete (dense) (exposed)        0.459861   
      Brick fired clay 1920                     0.265304   
Wall  Precast concrete (dense) (exposed)        0.477657   
      Brick fired clay 1920                     0.275571   
      Granite red                               0.762414   
Floor Oak beech ash walnut                      0.271144   

Material                                  Perlite board expanded  Rock wool  \
Roof  Precast concrete (dense) (exposed)                0.459861   0.459861   
      Brick fired clay 1920                             0.265304   0.265304   
Wall  Precast concrete (dense) (exposed)                0.477657   0.477657   
      Brick fired clay 1920                             0.275571   0.275571   
      Granite red                                       0.762414   0.762414   
Floor Oak beech ash walnut                              0.271144   0.271144   

Material                                  Urea formaldehyde resin foam  \
Roof  Precast concrete (dense) (exposed)                      0.459861   
      Brick fired clay 1920                                   0.265304   
Wall  Precast concrete (dense) (exposed)                      0.477657   
      Brick fired clay 1920                                   0.275571   
      Granite red                                             0.762414   
Floor Oak beech ash walnut                                    0.271144   

Material                                  Mineral wool  Polystyrene expanded  \
Roof  Precast concrete (dense) (exposed)      0.459861              0.459861   
      Brick fired clay 1920                   0.265304              0.265304   
Wall  Precast concrete (dense) (exposed)      0.477657              0.477657   
      Brick fired clay 1920                   0.275571              0.275571   
      Granite red                             0.762414              0.762414   
Floor Oak beech ash walnut                    0.271144              0.271144   

Material                                  Polyurethane foam  \
Roof  Precast concrete (dense) (exposed)           0.459861   
      Brick fired clay 1920                        0.265304   
Wall  Precast concrete (dense) (exposed)           0.477657   
      Brick fired clay 1920                        0.275571   
      Granite red                                  0.762414   
Floor Oak beech ash walnut                         0.271144   

Material                                  Polyisocyanurate aged with facers  
Roof  Precast concrete (dense) (exposed)                           0.459861  
      Brick fired clay 1920                                        0.265304  
Wall  Precast concrete (dense) (exposed)                           0.477657  
      Brick fired clay 1920                                        0.275571  
      Granite red                                                  0.762414  
Floor Oak beech ash walnut                                         0.271144